In [1]:
# Cell cycle scoring for ECs
 
import os
import warnings
import pandas as pd
import numpy as np
import scanpy as sc
import seaborn as sns
import matplotlib as mpl
import matplotlib.pyplot as plt
from pathlib import Path
from scipy.stats import mannwhitneyu
from statsmodels.stats.multitest import multipletests
 
warnings.filterwarnings("ignore")
 
# --- A. Paths ---
GLOBAL_OBJ_PATH = "/data/ag_reichart_lindberg/nhadizad/nastaran/projects/xeno/analysis/v1/cellchat/v1/merging_UMIs/Raw_UMI_merged_global_object_preclinical.h5ad"
RESULTS_DIR     = "/data/ag_reichart_lindberg/nhadizad/nastaran/projects/xeno/analysis/v1/cell_cycle_scoring/results"
SCORED_OBJ_PATH = os.path.join(RESULTS_DIR, "EC_cell_cycle_scored.h5ad")
 
# --- B. Metadata columns ---
CONDITION_COL  = "Condition"
SAMPLE_COL     = "Sample_ID"
CELL_TYPE_COL  = "cell_type"
CELL_STATE_COL = "cell_state"
 
# --- C. Target cell type ---
TARGET_CELL_TYPE = "EC"   
 
# --- D. Conditions ---
COND_ORDER = [
    "Control",
    "Prematurely-terminated",
    "3month-protocol",
    "6month-protocol",
    "PCMV/PRV",
    "Rejection",
]
 
COND_PALETTE = {
    "Control":                "#EFE48E",
    "3month-protocol":        "skyblue",
    "6month-protocol":        "#08528f",
    "Prematurely-terminated": "#C9B8D9",
    "PCMV/PRV":               "#F0C096",
    "Rejection":              "#E09E9E",
}


In [2]:

# --- E. Phase colors ---
PHASE_ORDER  = ["G1", "S", "G2M"]
PHASE_COLORS = {
    "G1":  "#66c2a5",   # teal
    "S":   "#fc8d62",   # orange
    "G2M": "#8da0cb",   # lavender
}
 
# --- F. Cell cycle gene lists ---
S_GENES = [
    "PCNA", "TYMS", "MCM2", "MCM4", "RRM1", "UNG", "GINS2", "MCM6", "CDCA7",
    "DTL", "UHRF1", "HELLS", "RFC2", "RPA2", "NASP", "RAD51AP1", "GMNN",
    "WDR76", "SLBP", "CCNE2", "UBR7", "POLD3", "MSH2", "ATAD2", "RAD51",
    "RRM2", "CDC45", "CDC6", "EXO1", "TIPIN", "DSCC1", "BLM", "CASP8AP2",
    "USP1", "CLSPN", "POLA1", "CHAF1B", "BRIP1", "E2F8",
]
 
G2M_GENES = [
    "HMGB2", "CDK1", "NUSAP1", "UBE2C", "BIRC5", "TPX2", "TOP2A", "NDC80",
    "CKS2", "NUF2", "MKI67", "TMPO", "CENPF", "TACC3", "SMC4", "CCNB2",
    "CKAP2L", "CKAP2", "AURKB", "BUB1", "KIF11", "ANP32E", "TUBB4B", "GTSE1",
    "KIF20B", "HJURP", "CDCA3", "CDC20", "TTK", "CDC25C", "KIF2C", "RANGAP1",
    "NCAPD2", "DLGAP5", "CDCA2", "CDCA8", "ECT2", "KIF23", "HMMR", "AURKA",
    "PSRC1", "ANLN", "LBR", "CKAP5", "CENPE", "CTCF", "NEK2", "G2E3",
    "GAS2L3", "CBX5", "CENPA",
]
 
# --- G. Visualization params ---
BOXPLOT_FIGSIZE     = (4.25, 3.25)
BOXPLOT_WIDTH       = 0.9
BOXPLOT_LINEWIDTH   = 0.9
MEDIAN_LINEWIDTH    = 0.7
STRIP_DOT_SIZE      = 4
STRIP_ALPHA         = 0.6
BRACKET_LINE_WIDTH  = 0.9
BRACKET_OFFSET_FRAC = 0.02
BRACKET_TICK_FRAC   = 0.001
STAR_GAP_FRAC       = 0.000001
STAR_FONT_SIZE      = 7
PIE_FIGSIZE         = (4, 4)   
PIE_AUTOPCT_MIN     = 0.5         
DPI                 = 300
 
# --- H. Global aesthetics ---
mpl.rcParams["pdf.fonttype"]       = 42
mpl.rcParams["ps.fonttype"]        = 42
mpl.rcParams["font.family"]        = "sans-serif"
mpl.rcParams["font.sans-serif"]    = ["Arial"]
mpl.rcParams["font.size"]          = 5
mpl.rcParams["font.weight"]        = "bold"
mpl.rcParams["axes.labelweight"]   = "bold"
mpl.rcParams["axes.titleweight"]   = "bold"
mpl.rcParams["figure.titleweight"] = "bold"
mpl.rcParams["axes.grid"]          = False
mpl.rcParams["axes.spines.top"]    = False
mpl.rcParams["axes.spines.right"]  = False
sns.set_style("ticks", {"axes.grid": False})
sc.settings.set_figure_params(dpi=300, dpi_save=300)
 
# Box style props
BOXPROPS     = dict(edgecolor="black", linewidth=BOXPLOT_LINEWIDTH)
MEDIANPROPS  = dict(color="black", linewidth=MEDIAN_LINEWIDTH)
WHISKERPROPS = dict(color="black", linewidth=BOXPLOT_LINEWIDTH)
CAPPROPS     = dict(color="black", linewidth=BOXPLOT_LINEWIDTH)
 
os.makedirs(RESULTS_DIR, exist_ok=True)
print("Configuration loaded.")


Configuration loaded.


# Cell cycle scoring

In [3]:
 
print("Loading global object...")
adata = sc.read(GLOBAL_OBJ_PATH)
print(f"  Full object: {adata.shape[0]} cells")
 
# Normalize
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)
 
# Subset to target cell type
adata_ec = adata[adata.obs[CELL_TYPE_COL] == TARGET_CELL_TYPE].copy()
print(f"  {TARGET_CELL_TYPE} cells: {adata_ec.n_obs}")
print(f"  Cell states: {sorted(adata_ec.obs[CELL_STATE_COL].unique().tolist())}")
 
# Uppercase var_names for gene matching
adata_ec.var_names = adata_ec.var_names.str.upper()
 
# Filter gene lists to those present
s_found   = [g for g in S_GENES if g in adata_ec.var_names]
g2m_found = [g for g in G2M_GENES if g in adata_ec.var_names]
print(f"  S-phase genes found: {len(s_found)}/{len(S_GENES)}")
print(f"  G2M-phase genes found: {len(g2m_found)}/{len(G2M_GENES)}")
 
# Score
sc.tl.score_genes_cell_cycle(adata_ec, s_genes=s_found, g2m_genes=g2m_found)
 
# Add indicator columns
adata_ec.obs["is_G1"]  = (adata_ec.obs["phase"] == "G1").astype(int)
adata_ec.obs["is_S"]   = (adata_ec.obs["phase"] == "S").astype(int)
adata_ec.obs["is_G2M"] = (adata_ec.obs["phase"] == "G2M").astype(int)
 
# Save
adata_ec.write(SCORED_OBJ_PATH)
print(f"  Scored object saved: {SCORED_OBJ_PATH}")
 
# Phase distribution
print("\n  Phase distribution:")
print(adata_ec.obs["phase"].value_counts())


Loading global object...
  Full object: 233200 cells
  EC cells: 9422
  Cell states: ['Arterial_EC', 'Cap_EC', 'Hypoxia_EC', 'ISG_cap_EC', 'Lymph_EC', 'Meso_EC', 'PCMV/PRV_EC', 'Prolif_EC', 'Ven_EC']
  S-phase genes found: 39/39
  G2M-phase genes found: 51/51
  Scored object saved: /data/ag_reichart_lindberg/nhadizad/nastaran/projects/xeno/analysis/v1/cell_cycle_scoring/results/EC_cell_cycle_scored.h5ad

  Phase distribution:
phase
G1     4810
S      3616
G2M     996
Name: count, dtype: int64


# Saving CSVs

In [4]:

adata_ec = sc.read(SCORED_OBJ_PATH)
csv_dir = os.path.join(RESULTS_DIR, "csvs")
os.makedirs(csv_dir, exist_ok=True)
 
score_cols = ["S_score", "G2M_score", "is_G1", "is_S", "is_G2M"]
 
# --- Master CSV 1: per sample × condition ---
df1 = adata_ec.obs[[SAMPLE_COL, CONDITION_COL] + score_cols].copy()
master_sample = (
    df1.groupby([SAMPLE_COL, CONDITION_COL], observed=True)
    .agg(
        N_cells=("S_score", "size"),
        S_score_median=("S_score", "median"),
        G2M_score_median=("G2M_score", "median"),
        Prop_G1=("is_G1", "mean"),
        Prop_S=("is_S", "mean"),
        Prop_G2M=("is_G2M", "mean"),
    )
    .reset_index()
    .round(4)
)
master_sample.to_csv(os.path.join(csv_dir, "master_per_sample.csv"), index=False)
 
# --- Master CSV 2: per sample × condition × cell_state ---
df2 = adata_ec.obs[[SAMPLE_COL, CONDITION_COL, CELL_STATE_COL] + score_cols].copy()
master_sample_state = (
    df2.groupby([SAMPLE_COL, CONDITION_COL, CELL_STATE_COL], observed=True)
    .agg(
        N_cells=("S_score", "size"),
        S_score_median=("S_score", "median"),
        G2M_score_median=("G2M_score", "median"),
        Prop_G1=("is_G1", "mean"),
        Prop_S=("is_S", "mean"),
        Prop_G2M=("is_G2M", "mean"),
    )
    .reset_index()
    .round(4)
)
master_sample_state.to_csv(os.path.join(csv_dir, "master_per_sample_and_state.csv"), index=False)
 
# --- Master CSV 3: per condition (aggregated medians) ---
master_condition = (
    master_sample.groupby(CONDITION_COL, observed=True)
    [["S_score_median", "G2M_score_median", "Prop_G1", "Prop_S", "Prop_G2M"]]
    .median()
    .reindex([c for c in COND_ORDER if c in master_sample[CONDITION_COL].unique()])
    .round(4)
)
master_condition.to_csv(os.path.join(csv_dir, "master_per_condition.csv"))
 
# --- Master CSV 4: per condition × cell_state ---
master_cond_state = (
    master_sample_state.groupby([CONDITION_COL, CELL_STATE_COL], observed=True)
    [["S_score_median", "G2M_score_median", "Prop_G1", "Prop_S", "Prop_G2M"]]
    .median()
    .round(4)
    .reset_index()
)
master_cond_state.to_csv(os.path.join(csv_dir, "master_per_condition_and_state.csv"), index=False)
 
print(f"All master CSVs saved to: {csv_dir}")
 
 

All master CSVs saved to: /data/ag_reichart_lindberg/nhadizad/nastaran/projects/xeno/analysis/v1/cell_cycle_scoring/results/csvs


# Boxplots per condition

In [5]:

adata_ec = sc.read(SCORED_OBJ_PATH)
box_dir = os.path.join(RESULTS_DIR, "boxplots", "global_by_condition")
os.makedirs(box_dir, exist_ok=True)
 
score_cols = ["S_score", "G2M_score", "is_G1", "is_S", "is_G2M"]
 
# Pseudobulk per sample
df = adata_ec.obs[[SAMPLE_COL, CONDITION_COL] + score_cols].copy()
grouped = df.groupby([SAMPLE_COL, CONDITION_COL], observed=True).mean(numeric_only=True).reset_index()
 
present_conds = [c for c in COND_ORDER if c in grouped[CONDITION_COL].unique()]
 
for phase in PHASE_ORDER:
    phase_col = f"is_{phase}"
    fig, ax = plt.subplots(figsize=BOXPLOT_FIGSIZE)
 
    sns.boxplot(
        data=grouped, x=CONDITION_COL, y=phase_col, order=present_conds,
        palette=COND_PALETTE, width=BOXPLOT_WIDTH, showfliers=False,
        boxprops=BOXPROPS, medianprops=MEDIANPROPS,
        whiskerprops=WHISKERPROPS, capprops=CAPPROPS, ax=ax,
    )
    sns.stripplot(
        data=grouped, x=CONDITION_COL, y=phase_col, order=present_conds,
        color="black", size=STRIP_DOT_SIZE, alpha=STRIP_ALPHA,
        linewidth=0, jitter=True, ax=ax,
    )
 
    ax.set_title(f"{TARGET_CELL_TYPE} — {phase} Phase Proportion by Condition",
                 fontweight="bold", fontsize=5)
    ax.set_ylabel(f"Proportion {phase}", fontweight="bold", fontsize=5)
    ax.set_xlabel("")
    plt.xticks(rotation=45, ha="right", fontweight="bold", fontsize=5)
    plt.yticks(fontweight="bold", fontsize=5)
    plt.tight_layout()
    plt.savefig(os.path.join(box_dir, f"{TARGET_CELL_TYPE}_{phase}_boxplot.pdf"),
                dpi=DPI, bbox_inches="tight")
    plt.close()
 
print(f"Global boxplots saved: {box_dir}")
 
 

Global boxplots saved: /data/ag_reichart_lindberg/nhadizad/nastaran/projects/xeno/analysis/v1/cell_cycle_scoring/results/boxplots/global_by_condition


# Boxplots per cell_state

In [6]:

adata_ec = sc.read(SCORED_OBJ_PATH)
state_box_dir = os.path.join(RESULTS_DIR, "boxplots", "per_cellstate")
os.makedirs(state_box_dir, exist_ok=True)
 
score_cols = ["is_G1", "is_S", "is_G2M"]
 
df = adata_ec.obs[[SAMPLE_COL, CONDITION_COL, CELL_STATE_COL] + score_cols].copy()
grouped = df.groupby([SAMPLE_COL, CONDITION_COL, CELL_STATE_COL], observed=True).mean(numeric_only=True).reset_index()
 
cell_states = sorted(grouped[CELL_STATE_COL].unique())
 
for cs in cell_states:
    cs_data = grouped[grouped[CELL_STATE_COL] == cs]
    cs_conds = [c for c in COND_ORDER if c in cs_data[CONDITION_COL].unique()]
 
    if len(cs_conds) < 2:
        continue
 
    safe_cs = cs.replace("/", "_").replace(" ", "_")
 
    # Melt to long format
    melted = cs_data.melt(
        id_vars=[SAMPLE_COL, CONDITION_COL],
        value_vars=score_cols,
        var_name="Phase", value_name="Proportion",
    )
    melted["Phase"] = melted["Phase"].map({"is_G1": "G1", "is_S": "S", "is_G2M": "G2M"})
 
    fig, ax = plt.subplots(figsize=BOXPLOT_FIGSIZE)
    sns.boxplot(
        data=melted, x=CONDITION_COL, y="Proportion", hue="Phase",
        order=cs_conds, hue_order=PHASE_ORDER, palette=PHASE_COLORS,
        width=BOXPLOT_WIDTH, showfliers=False,
        boxprops=BOXPROPS, medianprops=MEDIANPROPS,
        whiskerprops=WHISKERPROPS, capprops=CAPPROPS, ax=ax,
    )
 
    ax.set_title(f"{TARGET_CELL_TYPE} / {cs} — Cell Cycle Phases",
                 fontweight="bold", fontsize=5)
    ax.set_ylabel("Proportion", fontweight="bold", fontsize=5)
    ax.set_xlabel("")
    plt.xticks(rotation=45, ha="right", fontweight="bold", fontsize=5)
    plt.yticks(fontweight="bold", fontsize=5)
    ax.legend(bbox_to_anchor=(1.05, 1), loc="upper left", frameon=False,
              prop={"weight": "bold", "size": 5}, title="Phase")
    plt.tight_layout()
    plt.savefig(os.path.join(state_box_dir, f"{safe_cs}_phases_boxplot.pdf"),
                dpi=DPI, bbox_inches="tight")
    plt.close()
 
print(f"Per-state boxplots saved: {state_box_dir}")
 
 

Per-state boxplots saved: /data/ag_reichart_lindberg/nhadizad/nastaran/projects/xeno/analysis/v1/cell_cycle_scoring/results/boxplots/per_cellstate


# Pie charts

In [7]:
 
adata_ec = sc.read(SCORED_OBJ_PATH)
pie_dir = os.path.join(RESULTS_DIR, "pie_charts")
os.makedirs(pie_dir, exist_ok=True)
 
# Convert phase to string to avoid categorical issues throughout
adata_ec.obs["phase"] = adata_ec.obs["phase"].astype(str)
 
# Phase counts per cell state
phase_counts = (
    adata_ec.obs.groupby([CELL_STATE_COL, "phase"]).size().reset_index(name="count")
)
phase_counts["percent"] = (
    phase_counts.groupby(CELL_STATE_COL)["count"].transform(lambda x: 100 * x / x.sum())
)
 
# Master CSV for pie data
phase_counts.round(2).to_csv(os.path.join(pie_dir, "phase_distribution_all_states.csv"), index=False)
 
def autopct_fmt(pct):
    return f"{pct:.1f}%" if pct >= PIE_AUTOPCT_MIN else ""
 
# Individual pie per cell state
for cs in sorted(phase_counts[CELL_STATE_COL].unique()):
    sub = phase_counts[phase_counts[CELL_STATE_COL] == cs].copy()
 
    # Ensure all phases present, fill missing with 0
    full = pd.DataFrame({"phase": PHASE_ORDER})
    sub = full.merge(sub, on="phase", how="left").fillna({"count": 0, "percent": 0})
 
    safe_cs = cs.replace("/", "_").replace(" ", "_")
 
    fig, ax = plt.subplots(figsize=PIE_FIGSIZE)
    ax.pie(
        sub["percent"], labels=sub["phase"], autopct=autopct_fmt,
        startangle=90, counterclock=False,
        colors=[PHASE_COLORS[p] for p in sub["phase"]],
        wedgeprops=dict(linewidth=0.8, edgecolor="white"),
        textprops={"fontsize": 7, "fontweight": "bold"},
    )
    ax.set_title(f"{cs}", fontsize=9, fontweight="bold")
    ax.axis("equal")
    plt.tight_layout()
    plt.savefig(os.path.join(pie_dir, f"{safe_cs}_pie.pdf"), dpi=DPI, bbox_inches="tight")
    plt.close()
 
print(f"Pie charts + master CSV saved: {pie_dir}")
 


Pie charts + master CSV saved: /data/ag_reichart_lindberg/nhadizad/nastaran/projects/xeno/analysis/v1/cell_cycle_scoring/results/pie_charts


# Pie charts: Cell_state x Condition (just to check!)

In [8]:

adata_ec = sc.read(SCORED_OBJ_PATH)
pie_cond_dir = os.path.join(RESULTS_DIR, "pie_charts", "per_condition")
os.makedirs(pie_cond_dir, exist_ok=True)

# Convert phase to string and ensure categories are set
adata_ec.obs["phase"] = adata_ec.obs["phase"].astype(str)

# Calculate counts
phase_counts_cond = (
    adata_ec.obs.groupby([CONDITION_COL, CELL_STATE_COL, "phase"])
    .size()
    .reset_index(name="count")
)

# Calculate percentages per (Condition, State)
phase_counts_cond["percent"] = (
    phase_counts_cond.groupby([CONDITION_COL, CELL_STATE_COL])["count"]
    .transform(lambda x: 100 * x / x.sum())
)

# Master CSV
phase_counts_cond.round(2).to_csv(
    os.path.join(pie_cond_dir, "phase_distribution_per_condition_and_state.csv"), index=False
)

# Helper for labels
def autopct_fmt(pct):
    return f"{pct:.1f}%" if pct >= PIE_AUTOPCT_MIN else ""

for cs in sorted(phase_counts_cond[CELL_STATE_COL].unique()):
    cs_data = phase_counts_cond[phase_counts_cond[CELL_STATE_COL] == cs]
    # Filter conds to those present in our master config
    conds = [c for c in COND_ORDER if c in cs_data[CONDITION_COL].unique()]

    if not conds:
        continue

    safe_cs = cs.replace("/", "_").replace(" ", "_")
    ncols = min(3, len(conds))
    nrows = int(np.ceil(len(conds) / ncols))

    fig, axes = plt.subplots(nrows, ncols, figsize=(4 * ncols, 4 * nrows), squeeze=False)

    for i, cond in enumerate(conds):
        r, c_idx = divmod(i, ncols)
        ax = axes[r, c_idx]
        sub = cs_data[cs_data[CONDITION_COL] == cond].copy()

        # Ensure all phases present in the plot for consistency
        full = pd.DataFrame({"phase": PHASE_ORDER})
        sub = full.merge(sub, on="phase", how="left").fillna({"count": 0, "percent": 0})

        
        if sub["percent"].sum() > 0:
            ax.pie(
                sub["percent"], labels=sub["phase"], autopct=autopct_fmt,
                startangle=90, counterclock=False,
                colors=[PHASE_COLORS.get(p, 'grey') for p in sub["phase"]],
                wedgeprops=dict(linewidth=0.8, edgecolor="white"),
                textprops={"fontsize": 7, "fontweight": "bold"},
            )
            ax.set_title(f"{cond}", fontsize=7, fontweight="bold")
        else:
            ax.text(0, 0, "No Data", ha='center', fontsize=8)
            ax.set_title(f"{cond} (Empty)", fontsize=7)
        
        ax.axis("equal")

    # Hide empty subplots
    for j in range(len(conds), nrows * ncols):
        r, c_idx = divmod(j, ncols)
        axes[r, c_idx].axis("off")

    fig.suptitle(f"{cs} — Phase Distribution by Condition", fontsize=9, fontweight="bold")
    plt.tight_layout(rect=[0, 0, 1, 0.96]) # Adjust rect to fit suptitle
    plt.savefig(os.path.join(pie_cond_dir, f"{safe_cs}_pies_by_condition.pdf"),
                dpi=DPI, bbox_inches="tight")
    plt.close()

print(f"Condition-stratified pie charts saved: {pie_cond_dir}")

Condition-stratified pie charts saved: /data/ag_reichart_lindberg/nhadizad/nastaran/projects/xeno/analysis/v1/cell_cycle_scoring/results/pie_charts/per_condition
